In [ ]:
import os
import sys
import json
from tqdm import tqdm
from collections import defaultdict

In [ ]:
Path = "/home/user_name/U-MARVEL"
# 训练集候选池路径
train_cand_path = os.path.join(Path, "data/M-BEIR/cand_pool/global/mbeir_union_train_cand_pool.jsonl")
# 测试集候选池路径
union_test_cand_pool_path = os.path.join(Path,"data/M-BEIR/cand_pool/global/mbeir_union_test_cand_pool.jsonl")

In [ ]:
with open(train_cand_path, 'r', encoding='utf-8') as f:
    total_lines = sum(1 for _ in f)
print(total_lines)
with open(union_test_cand_pool_path, 'r', encoding='utf-8') as f:
    total_lines = sum(1 for _ in f)
print(total_lines)

In [ ]:
file_names = [
    "mbeir_cirr_task7_test.jsonl",
    "mbeir_edis_task2_test.jsonl",
    "mbeir_fashion200k_task0_test.jsonl",
    "mbeir_fashion200k_task3_test.jsonl",
    "mbeir_fashioniq_task7_test.jsonl",
    "mbeir_infoseek_task6_test.jsonl",
    "mbeir_infoseek_task8_test.jsonl",
    "mbeir_mscoco_task0_test.jsonl",
    "mbeir_mscoco_task3_test.jsonl",
    "mbeir_nights_task4_test.jsonl",
    "mbeir_oven_task6_test.jsonl",
    "mbeir_oven_task8_test.jsonl",
    "mbeir_visualnews_task0_test.jsonl",
    "mbeir_visualnews_task3_test.jsonl",
    "mbeir_webqa_task1_test.jsonl",
    "mbeir_webqa_task2_test.jsonl"
]
test_taskname2qid = defaultdict(list)
print(len(file_names))
total_lines = 0
Path_temp = "/home/user_name/U-MARVEL/data/M-BEIR/query/test"
for file_name in file_names:
    file_name = os.path.join(Path_temp,file_name)
    try:
        with open(file_name, 'r', encoding='utf-8') as file:
            lines = file.readlines()
            line_count = len(lines)
            total_lines += line_count
            parts = file_name.split("_")
            result = "_".join(parts[1:3])
            print(f"{file_name} 的行数: {line_count}")
    except FileNotFoundError:
        print(f"错误: 文件 {file_name} 未找到。")
    except Exception as e:
        print(f"错误: 读取文件 {file_name} 时发生未知错误: {e}")

print(f"所有文件的总行数: {total_lines}")

In [ ]:
query_union_train = "/home/user_name/U-MARVEL/data/M-BEIR/query/union_train/mbeir_union_up_train.jsonl"
with open(query_union_train, 'r', encoding='utf-8') as file:
    lines = file.readlines()
print(f"训练集所有文件的总行数: {len(lines)}")
print(f"测试集所有文件的总行数: {total_lines}")

In [ ]:
train_cand_path = os.path.join(Path, "data/M-BEIR/cand_pool/global/mbeir_union_train_cand_pool.jsonl")

In [ ]:
datasetid2name = {
    0: 'visualnews', 1: 'fashion200k', 2: 'webqa', 3: 'edis', 
    4: 'nights', 5: 'oven', 6: 'infoseek', 7: 'fashioniq', 8: 'cirr', 9: 'mscoco'}

In [ ]:
task_names = [
    'visualnews_task0', 'mscoco_task0', 'fashion200k_task0', 'webqa_task1', 'edis_task2', 
    'webqa_task2', 'visualnews_task3', 'mscoco_task3', 'fashion200k_task3', 'nights_task4', 
    'oven_task6', 'infoseek_task6', 'fashioniq_task7', 'cirr_task7', 'oven_task8', 'infoseek_task8']

In [ ]:
# 读取数据
taskname2qid = defaultdict(list)
query_union_train = "/home/user_name/U-MARVEL/data/M-BEIR/query/union_train/mbeir_union_up_train.jsonl"
with open(query_union_train, 'r', encoding='utf-8') as f:
    for line in tqdm(f, desc=f"Loading: "):
        item = json.loads(line)
        qid = item['qid']
        datasetid = int(item['qid'].split(":")[0])
        task_id = int(item["task_id"])
        taskname = datasetid2name[datasetid] + "_task" + str(task_id)
        taskname2qid[taskname].append(qid)

In [ ]:
num1,num2  = 0, 0
num3 = 0
for key,value in taskname2qid.items():
    print(f"{key} : {len((value))} {len((value))/1332790}")
    num1+=len((value))
    num3 += len((value))/1332790
    print(f"{key} : {len(set(value))}")
    num2+=len(set(value))
    print("*"*50)
print(num1)
print(num2)
print(num3)

### 每个任务单独抽取 1/10

In [ ]:
import random
random.seed(42)
taskname2qid_sample = defaultdict(list)
for task, qid_list in taskname2qid.items():
    sample_size = len(qid_list) // 10
    sampled_list = random.sample(qid_list, sample_size)
    taskname2qid_sample[task] = sampled_list

In [ ]:
num1,num2  = 0, 0
num3 = 0
for key,value in taskname2qid_sample.items():
    print(f"{key} : {len((value))} {len((value))/133276}")
    num1+=len((value))
    num3 += len((value))/133276
    print(f"{key} : {len(set(value))}")
    num2+=len(set(value))
    print("*"*50)
print(num1)
print(num2)
print(num3)

In [ ]:
all_sample_dids = []
for key,value in taskname2qid_sample.items():
    all_sample_dids.extend(value)
print(len(all_sample_dids))

In [ ]:
from collections import Counter
all_sample_dids = Counter(all_sample_dids)
all_sample_dids = dict(all_sample_dids)
print(len(all_sample_dids))

In [ ]:
# 读取数据
import copy
data_sample = []
query_union_train  = "/home/user_name/U-MARVEL/data/M-BEIR/query/union_train/mbeir_union_up_train.jsonl"
with open(query_union_train, 'r', encoding='utf-8') as f:
    # 统计文件行数
    line_count = sum(1 for _ in f)
    f.seek(0)  # 将文件指针移回文件开头
    for line in tqdm(f, total = line_count,desc=f"Loading: "):
        item = json.loads(line)
        qid  = item["qid"]
        if qid in all_sample_dids.keys() and all_sample_dids[qid]>0:
            data_sample.append(copy.deepcopy(item))
            all_sample_dids[qid]-=1
print(len(all_sample_dids)) 
print(len(data_sample))

In [ ]:
data_sample_file_path = os.path.join(Path, "data/M-BEIR/query/union_train/mbeir_union_up_train_10percent.jsonl")
with open(data_sample_file_path, 'w') as file:
    for item in data_sample:
        line = json.dumps(item) + '\n'
        file.write(line)
print(f"数据已成功保存到 {data_sample_file_path}")

#### 验证保存结果没有问题

In [ ]:
taskname2qid_10percent = defaultdict(list)
data_sample_file_path = os.path.join(Path, "data/M-BEIR/query/union_train/mbeir_union_up_train_10percent.jsonl")
with open(data_sample_file_path, 'r', encoding='utf-8') as f:
    # 统计文件行数
    line_count = sum(1 for _ in f)
    f.seek(0)  # 将文件指针移回文件开头
    for index,line in enumerate(tqdm(f, total = line_count,desc=f"Loading: ")):
        item = json.loads(line)
        qid = item['qid']
        datasetid = int(item['qid'].split(":")[0])
        task_id = int(item["task_id"])
        taskname = datasetid2name[datasetid] + "_task" + str(task_id)
        taskname2qid_10percent[taskname].append(qid)

In [ ]:
num1,num2  = 0, 0
num3 = 0
for key,value in taskname2qid_sample.items():
    print(f"{key} : {len((value))} {len((value))/133276}")
    num1+=len((value))
    num3 += len((value))/133276
    print(f"{key} : {len(set(value))}")
    num2+=len(set(value))
    print("*"*50)
print(num1)
print(num2)
print(num3)

#### 根据任务将qid 拆分成 各自的训练池

In [ ]:
import copy 
temp_taskname2qid_10percent = defaultdict(set)
taskname2data_10percent = defaultdict(list)
data_sample_file_path = os.path.join(Path, "data/M-BEIR/query/union_train/mbeir_union_up_train_10percent.jsonl")
with open(data_sample_file_path, 'r', encoding='utf-8') as f:
    # 统计文件行数
    line_count = sum(1 for _ in f)
    f.seek(0)  # 将文件指针移回文件开头
    for index,line in enumerate(tqdm(f, total = line_count,desc=f"Loading: ")):
        item = json.loads(line)
        qid = item['qid']
        datasetid = int(item['qid'].split(":")[0])
        task_id = int(item["task_id"])
        taskname = datasetid2name[datasetid] + "_task" + str(task_id)
        if qid not in temp_taskname2qid_10percent[taskname]:
            temp_taskname2qid_10percent[taskname].add(qid)
            taskname2data_10percent[taskname].append(copy.deepcopy(item))

In [ ]:
num = 0
for key,value in taskname2data_10percent.items():
    print(key,len(value))
    num+=len(value)
print(num)

In [ ]:
def save_jsonl_file(data, file_path):
    with open(file_path, 'w', encoding='utf-8') as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')

In [ ]:
base_file_name = "/home/user_name/U-MARVEL/data/M-BEIR/query/train/query_train/mbeir_train_"
file_name_list = []
for taskname,data in (taskname2data_10percent.items()):
    print(taskname,len(data))
    file_name = base_file_name + str(taskname)+ "_dedup_10percent.jsonl"
    save_jsonl_file(data,file_name)
    file_name_list.append(file_name)

In [ ]:
for file_name in (file_name_list):
    with open(file_name, 'r', encoding='utf-8') as f:
    # 统计文件行数
        line_count = sum(1 for _ in f)
    print(file_name)
    print(line_count)

### 每个任务各抽取 1/10

In [ ]:
import json
import random
import os
import tqdm
PATH = "/home/user_name/U-MARVEL"

In [ ]:
num1,num2 = 0,0
def sample_jsonl_file(file_path):
    random.seed(42)
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    sampled_data = random.sample(data, len(data) // 10)
    print(file_path)
    print(len(data),len(sampled_data))
    global num1,num2
    num1+= len(data)
    num2+= len(sampled_data)
    return sampled_data




In [ ]:
query_data_paths = [
    "data/M-BEIR/query/train/query_train/mbeir_train_mscoco_task0_dedup.jsonl",
    "data/M-BEIR/query/train/query_train/mbeir_train_mscoco_task3_dedup.jsonl",
    "data/M-BEIR/query/train/query_train/mbeir_train_cirr_task7_dedup.jsonl",
    "data/M-BEIR/query/train/query_train/mbeir_train_fashioniq_task7_dedup.jsonl",
    "data/M-BEIR/query/train/query_train/mbeir_train_webqa_task1_dedup.jsonl",
    "data/M-BEIR/query/train/query_train/mbeir_train_nights_task4_dedup.jsonl",
    "data/M-BEIR/query/train/query_train/mbeir_train_oven_task6_dedup.jsonl",
    "data/M-BEIR/query/train/query_train/mbeir_train_infoseek_task6_dedup.jsonl",
    "data/M-BEIR/query/train/query_train/mbeir_train_fashion200k_task0_dedup.jsonl",
    "data/M-BEIR/query/train/query_train/mbeir_train_fashion200k_task3_dedup.jsonl",
    "data/M-BEIR/query/train/query_train/mbeir_train_visualnews_task3_dedup.jsonl",
    "data/M-BEIR/query/train/query_train/mbeir_train_visualnews_task0_dedup.jsonl",
    "data/M-BEIR/query/train/query_train/mbeir_train_webqa_task2_dedup.jsonl",
    "data/M-BEIR/query/train/query_train/mbeir_train_oven_task8_dedup.jsonl",
    "data/M-BEIR/query/train/query_train/mbeir_train_infoseek_task8_dedup.jsonl",
    "data/M-BEIR/query/train/query_train/mbeir_train_edis_task2_dedup.jsonl",
]

sampled_files = []
for i, path in enumerate(query_data_paths):
    path = os.path.join(PATH,path)
    sampled_data = sample_jsonl_file(path)
    new_file_path = path.replace("_dedup.jsonl","_dedup_10percent.jsonl")
    save_jsonl_file(sampled_data, new_file_path)
    sampled_files.append(new_file_path)
print(num1)
print(num2)

In [ ]:
all_sampled_data = []
for file_path in sampled_files:
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            all_sampled_data.append(json.loads(line))
print(len(all_sampled_data))
random.shuffle(all_sampled_data)
# 保存合并并打乱后的数据到新文件
merged_file_path = "data/M-BEIR/query/train/query_train/mbeir_union_up_train_10percent.jsonl"
merged_file_path = os.path.join(PATH,merged_file_path)
save_jsonl_file(all_sampled_data, merged_file_path)
print(f"抽样并合并后的数据已保存到 {merged_file_path}")